# Garimpo gold and health (ADM2 panel)

Does upstream mercury-gold mining move health downstream? This notebook runs the gate
recorded as Evidence E2 and E4 in
[`output/research/mining_economics_leads.md`](../../research/mining_economics_leads.md):

1. the identifying variation in the upstream garimpo-gold share (MapBiomas Collection 11);
2. the **gold-share arm**: health outcomes on the gold share within 50 and 200 km
   upstream, specs a–d, then with the municipality's own (local) gold share added;
3. leave-one-out on Centro do Guilherme (MA);
4. the **Amazon mining-presence result** of [`02_analysis.ipynb`](02_analysis.ipynb) §9.3
   without the municipalities that ever have gold upstream and without the 2020 onsets,
   and the C11 composition of its switchers.

The estimators live in `src/analysis/garimpo_gold.py`: PPML with the outcome's
log-exposure offset, municipality and state×year FE, the baseline land-cover and climate
terms, Otto-3 clustering and the restricted wild-cluster score bootstrap (9,999 draws,
fixed seed). The C11 shares come from `adm2_panel` (`c11_*`, `local_c11_*`) in
percentage points. Tables are written to `output/analysis/health/garimpo_gold/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import warnings

import numpy as np
import pandas as pd

from src.analysis import garimpo_gold as G
from src.analysis import health as H

warnings.filterwarnings("ignore", message=r".*singleton fixed effect")
warnings.filterwarnings("ignore", message=r".*removed because of separation")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:.4g}")

OUT = PROJECT_ROOT / "output" / "analysis" / "health" / "garimpo_gold"
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
df = G.build_gate_frame()
print(f"{len(df):,} municipality-years · {df.mun_id.nunique():,} municipalities · {df.year.min()}–{df.year.max()}")
for radius in G.GOLD_RADII:
    sample = G.gold_sample(df, radius)
    amazon = df[df.mun_id.isin(sample) & (df.amazon == 1)].mun_id.nunique()
    basins = df[df.mun_id.isin(sample)].otto_basin_l3.nunique()
    print(f"gold within {radius} km in 2000–2019: {len(sample)} municipalities ({amazon} Amazon), {basins} Otto-3 basins")

138,912 municipality-years · 5,562 municipalities · 2000–2024
gold within 50 km in 2000–2019: 135 municipalities (64 Amazon), 39 Otto-3 basins


gold within 200 km in 2000–2019: 291 municipalities (107 Amazon), 57 Otto-3 basins


## 1. Identifying variation

Within the gold samples, how many municipalities see their upstream gold share change
over 2000–2019, and by how much (within-SD after municipality and state×year FE)? And how
much of the gold near these municipalities lies in their own trenches, which the upstream
arm does not use?

In [3]:
variation = G.gold_variation(df)
own = G.own_vs_upstream_gold(df)
variation.to_csv(OUT / "variation.csv", index=False)
own.to_csv(OUT / "own_vs_upstream.csv", index=False)
display(variation)
display(own)

,R,subset,municipalities,otto3_basins,any change 2000-19,rise >= 0.01 pp,rise >= 0.1 pp,fall <= -0.01 pp,within SD gold (pp),within SD gold_active (pp)
0,50,all,135,39,129,42,16,2,0.06114,0.05986
1,50,Amazon,64,26,60,32,15,0,0.08763,0.08598
2,200,all,291,57,289,48,17,2,0.03914,0.03829
3,200,Amazon,107,40,106,39,17,0,0.06426,0.06292


,R,own-trench gold px,upstream gold px (0-R km),own share,municipalities with own gold,with upstream gold but no own gold
0,50,4.86e+07,2.589e+07,0.6524,61,74
1,200,4.914e+07,7.364e+07,0.4002,72,219


**Reading the results.** 135 municipalities have gold within 50 km upstream in
2000–2019 (64 Amazon, 39 basins), 291 within 200 km (107 Amazon, 57 basins). The share
changes in almost all of them, but only 32 Amazon municipalities gain at least 0.01 pp at
50 km (39 at 200 km), and the within-SD is 0.04–0.09 pp. 65% of the gold pixels in the
50 km sample lie in the municipalities' own trenches (40% at 200 km), which the upstream
arm does not use.

## 2. Gold-share arm

Specs at each radius R, on the municipalities with gold within R in 2000–2019:

- **a** pooled, 2000–2019;
- **b** separate Amazon and other-biome slopes, 2000–2019;
- **c** the active (exposed-soil) gold share, 2000–2019;
- **d** pooled, 2000–2024 (across the 2020 mapping step).

Controls: other C11 mining within R, the local C10 mining terms and the baseline terms.
Estimates are per pp of gold share; `pct_per_sd` is the effect of one within-SD. Specs a
and b are bootstrapped.

In [4]:
arm = pd.concat([G.gold_arm(df, radius) for radius in G.GOLD_RADII], ignore_index=True)
arm.to_csv(OUT / "gold_arm.csv", index=False)
gold_terms = arm[~arm["term"].str.startswith("other_mining")]
display(gold_terms.pivot_table(index=["outcome"], columns=["R", "spec", "term"], values="pct_per_sd", sort=False).round(1))
display(gold_terms.pivot_table(index=["outcome"], columns=["R", "spec", "term"], values="p_wcb", sort=False).dropna(axis=1, how="all").round(3))

R                             50                                                                                    200                                                                      
spec             a pooled 2000-19 b by region 2000-19                c active 2000-19 d pooled 2000-24 a pooled 2000-19 b by region 2000-19                 c active 2000-19 d pooled 2000-24
term                      gold_50     gold_50__amazon gold_50__other   gold_active_50          gold_50         gold_200    gold_200__amazon gold_200__other  gold_active_200         gold_200
outcome                                                                                                                                                                                      
nervous_system                2.2                 4.1           -8.7              1.5                4                3                 5.2            -4.6              2.7              1.6
blood                         9.3                  13            9.3              6.8                4              5.3                 8.9             4.9                4              3.5
renal_urinary                 4.4                 5.9              5                3              3.4              5.6                 9.3             3.3                5              4.1
toxic_poisoning               6.5                10.7           -6.7              4.6              6.3              8.7                15.4            -6.1              8.3              5.8
congenital                    9.3                  14           -6.3              8.8             11.3                7                11.8            -0.9              6.4              6.1
perinatal                     1.4                 3.3          -14.9             -0.1              0.2              3.3                 5.4            -3.1              2.2              0.7
low_birth_weight              1.2                 1.7             -1              0.8              1.3              0.6                   1            -0.7              0.4              1.1
preterm                      -0.6                -0.9            3.9             -0.6              0.4             -0.3                -0.5             2.5             -0.4              0.3
infant_deaths                 2.1                 3.1           -1.1              2.3              1.4              1.5                 2.5             0.8              1.5              0.7
injury_poisoning              3.5                 5.2           -2.1              1.5              5.2              5.3                 8.8             1.7              4.3              3.6
injuries                      3.7                 5.5           -1.3              1.4              5.7              5.4                   9             2.9              4.2              3.8

R                             50                                                  200                                    
spec             a pooled 2000-19 b by region 2000-19                a pooled 2000-19 b by region 2000-19                
term                      gold_50     gold_50__amazon gold_50__other         gold_200    gold_200__amazon gold_200__other
outcome                                                                                                                  
nervous_system              0.376               0.158          0.383            0.212               0.153           0.487
blood                       0.006               0.006          0.462            0.088               0.082            0.38
renal_urinary               0.109               0.119          0.632            0.015               0.017           0.564
toxic_poisoning             0.084               0.056          0.377            0.031                0.02           0.227
congenital                   0.06                0.06          0.321            0.065               0.062            0.69
perinatal                   0.893               0.867          0.065             0.37               0.355           0.686
low_birth_weight            0.134               0.121          0.205            0.183               0.171           0.229
preterm                     0.524               0.517          0.242            0.508               0.492           0.246
infant_deaths               0.113               0.095          0.665             0.01               0.011           0.617
injury_poisoning            0.099               0.087          0.446            0.005               0.005           0.481
injuries                    0.154               0.146          0.595            0.004               0.004           0.256

In [5]:
headline = gold_terms[gold_terms["spec"].str[0].isin(["a", "d"])]
display(headline[["R", "spec", "outcome", "estimate_per_pp", "se_per_pp", "within_sd_pp", "pct_per_sd",
                  "p_analytic", "p_wcb", "n_mun", "n_clusters", "n_obs"]])

,R,spec,outcome,estimate_per_pp,se_per_pp,within_sd_pp,pct_per_sd,p_analytic,p_wcb,n_mun,n_clusters,n_obs
0,50,a pooled 2000-19,nervous_system,0.3597,0.3741,0.06114,2.223,0.3364,0.376,133,38,2659
2,50,a pooled 2000-19,blood,1.448,0.8282,0.06114,9.255,0.08048,0.005901,133,38,2659
4,50,a pooled 2000-19,renal_urinary,0.705,0.413,0.06114,4.405,0.08779,0.1085,133,38,2659
6,50,a pooled 2000-19,toxic_poisoning,1.023,0.4755,0.06114,6.458,0.03138,0.08391,133,38,2650
8,50,a pooled 2000-19,congenital,1.448,0.429,0.06114,9.259,0.0007361,0.06021,133,38,2659
10,50,a pooled 2000-19,perinatal,0.2285,0.8696,0.06114,1.407,0.7927,0.8927,133,38,2659
12,50,a pooled 2000-19,low_birth_weight,0.1909,0.2044,0.06114,1.174,0.3503,0.1342,133,38,2659
14,50,a pooled 2000-19,preterm,-0.09865,0.1437,0.06114,-0.6014,0.4924,0.5237,133,38,1064
16,50,a pooled 2000-19,infant_deaths,0.3445,0.1137,0.06114,2.129,0.002439,0.1127,133,38,2658
18,50,a pooled 2000-19,injury_poisoning,0.5575,0.3688,0.06114,3.468,0.1306,0.09881,133,38,2659


**Reading the results.** Per within-SD of gold share (2000–2019, spec a), infant deaths
rise 2.1% at 50 km (boot p 0.11) and 1.5% at 200 km (0.010); congenital hospitalizations
9.3% and 7.0% (0.06, 0.065); toxic poisoning 6.5% and 8.7% (0.08, 0.031). The **injury
placebos rise as much** (3.5–5.4%), with the strongest bootstrap p-values at 200 km
(0.005, 0.004). LBW, preterm and perinatal are null. On 2000–2024 (spec d) infant deaths
fall to a third (0.11 per pp, p = 0.24) and the placebos become significant analytically.
The Amazon slopes (spec b) are 1.5–1.7 times the pooled per-SD effects with the same
pattern; the other-biome slopes have almost no variation.

## 3. Local gold

Specs a and b with the municipality's own-trench gold share (`local_gold`, the negative
buckets of the trenches touching it, as `local_lc_mining`) added. Local gold is
confounded with the local mining economy (income, migration, injuries), so this asks
whether the upstream term is picking up gold in the municipality itself, not what local
gold does.

In [6]:
local = pd.concat([G.gold_arm(df, radius, local_gold=True) for radius in G.GOLD_RADII], ignore_index=True)
local.to_csv(OUT / "gold_arm_local_gold.csv", index=False)
for radius in G.GOLD_RADII:
    sample = df[df.mun_id.isin(G.gold_sample(df, radius)) & (df.year <= G.GATE_LAST_YEAR)]
    moving = sample.groupby("mun_id")[G.LOCAL_GOLD].nunique().gt(1).sum()
    print(f"R={radius}: local gold within-SD {G.within_sd(sample, G.LOCAL_GOLD):.3g} pp, "
          f"{moving} municipalities with changing local gold")
shown = local[~local["term"].str.startswith("other_mining")]
display(shown.pivot_table(index="outcome", columns=["R", "spec", "term"], values="estimate_per_pp", sort=False).round(3))
display(shown.pivot_table(index="outcome", columns=["R", "spec", "term"], values="p_wcb", sort=False).round(3))

R=50: local gold within-SD 0.119 pp, 58 municipalities with changing local gold
R=200: local gold within-SD 0.0829 pp, 68 municipalities with changing local gold


R                                          50                                                                                                  200                                                                       
spec             a pooled 2000-19 + local gold            b by region 2000-19 + local gold                           a pooled 2000-19 + local gold            b by region 2000-19 + local gold                           
term                                   gold_50 local_gold                  gold_50__amazon gold_50__other local_gold                      gold_200 local_gold                 gold_200__amazon gold_200__other local_gold
outcome                                                                                                                                                                                                                  
nervous_system                           0.399     -0.105                            0.558         -6.782     -0.259                         0.691      0.121                            0.744          -9.779      0.074
blood                                    1.653     -0.506                             1.57          6.073     -0.418                         1.276      0.059                            1.256            10.2      0.091
renal_urinary                            0.666      0.099                             0.59          3.599      0.156                         1.412     -0.022                            1.387           6.816          0
toxic_poisoning                          1.055     -0.207                            1.233          -5.21      -0.46                         2.084      0.148                            2.214          -13.12       0.05
congenital                               1.314      0.347                            1.412         -4.391      0.213                         1.682      0.083                            1.697          -1.778      0.067
perinatal                                0.404     -0.503                            0.627         -12.07     -0.719                         1.032     -0.399                            1.059           -7.23     -0.439
low_birth_weight                         0.142      0.197                            0.149         -0.551      0.185                         0.074      0.222                            0.076          -0.945      0.218
preterm                                 -0.054     -0.273                           -0.061           2.55     -0.236                        -0.015     -0.319                           -0.016           4.396     -0.296
infant_deaths                            0.327      0.077                            0.337         -0.783      0.059                         0.365      0.052                            0.362           1.866      0.058
injury_poisoning                         0.647     -0.291                            0.681          -1.77     -0.335                         1.358      -0.07                            1.354           3.326     -0.063
injuries                                 0.685      -0.28                             0.71          -1.22     -0.313                         1.353     -0.033                            1.348           5.862     -0.018

R                                          50                                                                                                  200                                                                       
spec             a pooled 2000-19 + local gold            b by region 2000-19 + local gold                           a pooled 2000-19 + local gold            b by region 2000-19 + local gold                           
term                                   gold_50 local_gold                  gold_50__amazon gold_50__other local_gold                      gold_200 local_gold                 gold_200__amazon gold_200__other local_gold
outcome                                                                                                                                                                                                                  
nervous_system                           0.399       0.71                            0.149           0.37      0.388                         0.261      0.589                            0.191           0.494      0.751
blood                                     0.01       0.48                            0.011           0.51       0.57                         0.116      0.664                            0.123           0.369      0.463
renal_urinary                            0.202      0.911                            0.242          0.647       0.85                          0.02      0.928                            0.025           0.568          1
toxic_poisoning                          0.099      0.811                             0.06          0.334      0.569                          0.04       0.69                            0.026           0.228      0.889
congenital                               0.122      0.709                              0.1          0.294      0.798                         0.085      0.685                            0.082           0.702      0.744
perinatal                                0.773      0.539                            0.625          0.059      0.396                         0.261      0.503                            0.236           0.654      0.463
low_birth_weight                         0.284      0.262                            0.264          0.274      0.279                         0.424      0.117                            0.408           0.341      0.113
preterm                                  0.654      0.485                            0.608          0.308      0.535                         0.906      0.227                             0.89           0.309      0.247
infant_deaths                            0.128       0.82                            0.107          0.724      0.875                         0.022      0.654                            0.028             0.6      0.645
injury_poisoning                         0.068      0.333                            0.051          0.347       0.27                         0.009      0.702                             0.01           0.484      0.722
injuries                                 0.073      0.404                            0.058          0.477      0.352                         0.009      0.845                             0.01           0.248      0.912

In [7]:
# Upstream gold with and without local gold, spec a.
compare = pd.concat([
    arm[arm["spec"].eq("a pooled 2000-19") & arm["term"].str.match(r"gold_\d+$")].assign(local_gold="no"),
    local[local["spec"].str.startswith("a ") & local["term"].str.match(r"gold_\d+$")].assign(local_gold="yes"),
])
display(compare.pivot_table(index="outcome", columns=["R", "local_gold"], values=["estimate_per_pp", "p_wcb"],
                            sort=False).round(3))

estimate_per_pp                      p_wcb                  
R                            50     200    50     200   50    200   50    200
local_gold                    no     no    yes    yes    no    no   yes   yes
outcome                                                                      
nervous_system              0.36  0.757  0.399  0.691 0.376 0.212 0.399 0.261
blood                      1.448  1.322  1.653  1.276 0.006 0.088  0.01 0.116
renal_urinary              0.705  1.397  0.666  1.412 0.109 0.015 0.202  0.02
toxic_poisoning            1.023  2.134  1.055  2.084 0.084 0.031 0.099  0.04
congenital                 1.448  1.735  1.314  1.682  0.06 0.065 0.122 0.085
perinatal                  0.229  0.819  0.404  1.032 0.893  0.37 0.773 0.261
low_birth_weight           0.191  0.154  0.142  0.074 0.134 0.183 0.284 0.424
preterm                   -0.099 -0.087 -0.054 -0.015 0.524 0.508 0.654 0.906
infant_deaths              0.345  0.383  0.327  0.365 0.113  0.01 0.128 0.022
injury_poisoning           0.558  1.316  0.647  1.358 0.099 0.005 0.068 0.009
injuries                   0.595  1.332  0.685  1.353 0.154 0.004 0.073 0.009

**Reading the results.** Local gold moves within 58 municipalities of the 50 km
sample (within-SD 0.12 pp). Adding it barely changes the upstream gold coefficients
(infant deaths 0.345 → 0.327 per pp at 50 km, 0.383 → 0.365 at 200 km; injury placebos
0.56 → 0.65 and 1.32 → 1.36), and local gold itself has no significant coefficient for
any outcome (boot p ≥ 0.11). The upstream result is therefore not gold in the
municipality itself, and the failed placebo stays.

## 4. Leave-one-out: Centro do Guilherme

Centro do Guilherme (MA, `210315`) has the largest gold-share change in the 50 km
sample. Specs a and b on 2000–2019 without it.

In [8]:
loo = G.leave_one_out(df)
loo.to_csv(OUT / "leave_one_out_210315.csv", index=False)
display(loo)

,R,spec,term,outcome,estimate_per_pp,se,p_analytic,p_wcb,n_mun,n_clusters
0,50,a pooled,gold_50,infant_deaths,0.1642,0.3464,0.6355,0.6197,132,38
1,50,a pooled,gold_50,congenital,1.877,0.7988,0.01882,0.08501,132,38
2,50,a pooled,gold_50,blood,2.732,0.6539,2.933e-05,0.005201,132,38
3,50,a pooled,gold_50,toxic_poisoning,1.06,0.7804,0.1745,0.2643,132,38
4,50,a pooled,gold_50,injury_poisoning,1.048,0.4546,0.02111,0.1908,132,38
5,50,a pooled,gold_50,injuries,1.265,0.614,0.03933,0.1986,132,38
6,50,b by region,gold_50__amazon,infant_deaths,0.1843,0.3434,0.5914,0.5715,132,38
7,50,b by region,gold_50__amazon,congenital,1.983,0.8882,0.02555,0.08241,132,38
8,50,b by region,gold_50__amazon,blood,2.663,0.6245,2.001e-05,0.005201,132,38
9,50,b by region,gold_50__amazon,toxic_poisoning,1.269,0.7954,0.1106,0.1866,132,38


**Reading the results.** Without Centro do Guilherme, the 50 km infant-death
coefficient falls from 0.34 to 0.16 per pp (boot p 0.62); blood stays (2.7, p 0.005).
At 200 km the infant-death coefficient rises to 0.54 (p 0.08), but the placebos rise as
well (injury and poisoning 2.4 per pp, p 0.004).

## 5. The Amazon mining-presence result

The §9.3 coefficient of [`02_analysis.ipynb`](02_analysis.ipynb): C10 mining appearing
within 50 km upstream in the Amazon → infant deaths, with the injury placebo. Samples:

- **a** as in §9.3;
- **b, c** without every municipality that ever has C11 garimpo gold within 50 / 200 km;
- **e, f** without the switchers whose onset is 2020 / 2020 or later (the C10 mapping step);
- **g** 2000–2019 only; **h** without 2020–21.

In [9]:
presence = G.presence_spec(df)
presence.to_csv(OUT / "presence.csv", index=False)
display(presence)

,sample,outcome,estimate,se,p_analytic,p_wcb,n_mun,n_amazon_mun,n_amazon_switchers,n_amazon_switchers_exposed,n_amazon_switcher_basins,n_clusters,n_obs
0,a original,infant_deaths,0.2074,0.05257,7.962e-05,0.0015,1189,169,36,36,18,115,29675
1,a original,injury_poisoning,0.07424,0.05455,0.1735,0.4387,1189,169,36,36,18,115,29681
2,b drop gold_50 > 0,infant_deaths,0.212,0.05032,2.516e-05,0.0003,1054,104,29,29,16,97,26302
3,b drop gold_50 > 0,injury_poisoning,0.08883,0.06487,0.1709,0.4684,1054,104,29,29,16,97,26307
4,c drop gold_200 > 0,infant_deaths,0.2182,0.05329,4.236e-05,0.0005001,962,92,27,27,15,91,24002
5,c drop gold_200 > 0,injury_poisoning,0.08429,0.06503,0.1949,0.4955,962,92,27,27,15,91,24007
6,e drop switchers with onset 2020,infant_deaths,0.2591,0.05203,6.387e-07,0.005801,1178,158,25,25,14,114,29400
7,e drop switchers with onset 2020,injury_poisoning,-0.05268,0.03868,0.1731,0.177,1178,158,25,25,14,114,29406
8,f drop switchers with onset >= 2020,infant_deaths,0.2628,0.05334,8.341e-07,0.008901,1171,151,18,18,11,111,29225
9,f drop switchers with onset >= 2020,injury_poisoning,-0.06109,0.04134,0.1395,0.1596,1171,151,18,18,11,111,29231


In [10]:
composition = G.switcher_composition(df)
composition.to_csv(OUT / "switcher_composition.csv", index=False)
print("largest C11 group within 50 km one year after onset:", composition["largest_c11_group"].value_counts().to_dict())
print("no C11 within 50 km at onset + 1:", int((composition["c11_total_50"] == 0).sum()),
      "| none in any year:", int((~composition["c11_any_year_50"]).sum()),
      f"| none within {G.GOLD_RADII[1]} km at onset + 1:", int((composition[f"c11_at_year_{G.GOLD_RADII[1]}"] == 0).sum()))
display(composition)

largest C11 group within 50 km one year after onset: {'garimpo_unknown': 14, 'garimpo_other': 7, 'industrial_other': 7, 'garimpo_gold': 6, 'none': 2}
no C11 within 50 km at onset + 1: 2 | none in any year: 2 | none within 200 km at onset + 1: 2


,mun_id,onset,year_used,basin,c10_upstream_pp_50,garimpo_gold,garimpo_unknown,garimpo_other,industrial_gold,industrial_other,c11_total_50,largest_c11_group,c11_any_year_50,c11_at_year_200,gold_any_year_200
0,150500,2020,2021,711,0.5111,0,0.08634,0.4729,0,0,0.5592,garimpo_other,True,0.5592,False
1,210315,2007,2008,716,0.2577,0.2089,0,0.1776,0,0,0.3865,garimpo_gold,True,0.3865,True
2,150080,2020,2021,612,0.2023,0,0.06691,0.1396,0,0,0.2065,garimpo_other,True,0.1082,False
3,130006,2020,2021,497,0.1373,0,0.1289,0,0,0,0.1289,garimpo_unknown,True,0.0569,False
4,210637,2007,2008,716,0.03246,0.03027,0,0.02237,0,0,0.05264,garimpo_gold,True,0.05264,True
5,130390,2020,2021,497,0.03196,0,0.03101,0,0,0,0.03101,garimpo_unknown,True,0.01369,False
6,210317,2020,2021,714,0.01055,0.01708,0,0,0,0,0.01708,garimpo_gold,True,0.007539,True
7,150690,2020,2021,711,0.01604,0,0.01628,0,0,0,0.01628,garimpo_unknown,True,0.5867,False
8,211227,2020,2021,722,0.01316,0,0.01613,0,0,0,0.01613,garimpo_unknown,True,0.007121,False
9,150620,2020,2021,711,0.01492,0,0.01514,0,0,0,0.01514,garimpo_unknown,True,0.0623,False


**Reading the results.** The presence result is not gold. Dropping every
municipality that ever has gold within 50 km (29 switchers left) or 200 km (27) leaves
the Amazon infant-death coefficient at +0.21 and +0.22 (boot p 0.0003, 0.0005), and the
injury placebo flat. It also survives the 2020 mapping step: +0.26 without the 2020
onsets (p 0.006) and without onsets from 2020 on (p 0.009), +0.27 on 2000–2019 (p 0.013),
+0.21 without 2020–21 (p 0.005). One year after onset, the largest C11 group within
50 km is garimpo of unknown substance for 14 of the 36 switchers, other garimpo for 7,
non-gold industrial for 7 and garimpo gold for 6 (all in Maranhão); 2 have no C11 mining.

## 6. Reproduction check

The estimates of the run recorded in the leads note (E2, E4), to the precision kept there.

In [11]:
def pick(table, **conditions):
    mask = np.logical_and.reduce([table[column] == value for column, value in conditions.items()])
    (row,) = table[mask].to_dict("records")
    return row

reference = [
    ("presence a, infant deaths: estimate", pick(presence, sample="a original", outcome="infant_deaths")["estimate"], 0.207407),
    ("presence a, infant deaths: boot p", pick(presence, sample="a original", outcome="infant_deaths")["p_wcb"], 0.0015),
    ("presence b, infant deaths: boot p", pick(presence, sample="b drop gold_50 > 0", outcome="infant_deaths")["p_wcb"], 0.0003),
    ("presence e, infant deaths: estimate", pick(presence, sample="e drop switchers with onset 2020", outcome="infant_deaths")["estimate"], 0.259084),
    ("gold_50 a, infant deaths: per pp", pick(arm, R=50, spec="a pooled 2000-19", term="gold_50", outcome="infant_deaths")["estimate_per_pp"], 0.344536),
    ("gold_50 a, infant deaths: boot p", pick(arm, R=50, spec="a pooled 2000-19", term="gold_50", outcome="infant_deaths")["p_wcb"], 0.112711),
    ("gold_200 a, infant deaths: boot p", pick(arm, R=200, spec="a pooled 2000-19", term="gold_200", outcome="infant_deaths")["p_wcb"], 0.010301),
    ("gold_200 a, injury_poisoning: boot p", pick(arm, R=200, spec="a pooled 2000-19", term="gold_200", outcome="injury_poisoning")["p_wcb"], 0.0045),
    ("gold_50 d, infant deaths: per pp", pick(arm, R=50, spec="d pooled 2000-24", term="gold_50", outcome="infant_deaths")["estimate_per_pp"], 0.111399),
    ("leave-one-out 50 a, infant deaths: per pp", pick(loo, R=50, spec="a pooled", outcome="infant_deaths")["estimate_per_pp"], 0.164206),
    ("leave-one-out 200 a, injury_poisoning: per pp", pick(loo, R=200, spec="a pooled", outcome="injury_poisoning")["estimate_per_pp"], 2.396221),
]
check = pd.DataFrame(reference, columns=["quantity", "this run", "recorded"])
check["match"] = np.isclose(check["this run"], check["recorded"], rtol=1e-4, atol=1e-6)
display(check)
assert check["match"].all(), "Reproduction mismatch"
print(f"Largest C11 group counts: {composition['largest_c11_group'].value_counts().to_dict()}")

,quantity,this run,recorded,match
0,"presence a, infant deaths: estimate",0.2074,0.2074,True
1,"presence a, infant deaths: boot p",0.0015,0.0015,True
2,"presence b, infant deaths: boot p",0.0003,0.0003,True
3,"presence e, infant deaths: estimate",0.2591,0.2591,True
4,"gold_50 a, infant deaths: per pp",0.3445,0.3445,True
5,"gold_50 a, infant deaths: boot p",0.1127,0.1127,True
6,"gold_200 a, infant deaths: boot p",0.0103,0.0103,True
7,"gold_200 a, injury_poisoning: boot p",0.0045,0.0045,True
8,"gold_50 d, infant deaths: per pp",0.1114,0.1114,True
9,"leave-one-out 50 a, infant deaths: per pp",0.1642,0.1642,True


Largest C11 group counts: {'garimpo_unknown': 14, 'garimpo_other': 7, 'industrial_other': 7, 'garimpo_gold': 6, 'none': 2}
